# M03-P02 — Attention & Tiny Transformer

En M03-P01 abrimos el aprendizaje de una red neuronal. Ahora abriremos otro mecanismo fundamental de los modelos modernos: **self-attention**.

Queremos observar de forma tangible:

```text
texto
→ tokens
→ IDs
→ embeddings
→ posiciones
→ Q / K / V
→ scores
→ softmax
→ matriz de atención
→ representación contextual
```

Después entrenaremos un **Tiny Transformer** muy pequeño para comprobar cómo cambia la atención cuando el contexto cambia.

> Esta práctica no construye un Transformer completo desde cero ni pretende reproducir un LLM. Usamos una versión mínima y observable para entender las piezas.

## 1. Preparar el laboratorio

Todo se ejecuta localmente en CPU. No usamos AWS ni Bedrock.

In [ ]:
import math
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

## 2. Un pequeño vocabulario empresarial

Para poder abrir todas las piezas sin depender de un tokenizer externo, crearemos un corpus sintético muy pequeño.

La tarea posterior será clasificar frases como:

```text
vpn funciona
vpn no funciona
crm está disponible
crm está caído
```

en dos clases:

```text
0 = servicio saludable
1 = incidente
```

El corpus es deliberadamente sencillo: queremos estudiar **attention**, no construir un clasificador empresarial de producción.

In [ ]:
services = ["vpn", "crm", "erp", "correo", "portal", "wifi", "dns", "proxy"]

incident_templates = [
    "{s} no responde",
    "{s} no funciona",
    "{s} está caído",
    "{s} presenta error",
    "{s} falla",
    "{s} está indisponible",
    "{s} tiene fallos",
]

healthy_templates = [
    "{s} responde correctamente",
    "{s} funciona correctamente",
    "{s} está disponible",
    "{s} está operativo",
    "{s} sin errores",
    "{s} funciona",
    "{s} sí funciona",
]

samples = []
for service in services:
    for template in incident_templates:
        samples.append((template.format(s=service), 1))
    for template in healthy_templates:
        samples.append((template.format(s=service), 0))

df = pd.DataFrame(samples, columns=["text", "label"])
df["label_name"] = df["label"].map({0: "healthy", 1: "incident"})

print("Ejemplos:", len(df))
display(df.head(10))
display(df["label_name"].value_counts())

## 3. Tokenización: texto → tokens → IDs

Un modelo no recibe directamente palabras. Necesita números.

Usaremos un tokenizer muy simple para que todo sea visible:

1. pasamos a minúsculas;
2. separamos palabras;
3. añadimos `[CLS]` al principio;
4. convertimos cada token a un ID;
5. rellenamos con `[PAD]` hasta una longitud fija.

`[CLS]` será un token especial cuya representación final utilizaremos para clasificar la frase completa.

In [ ]:
def tokenize(text):
    return re.findall(r"[\wáéíóúñü]+", text.lower(), flags=re.UNICODE)

all_tokens = sorted({
    token
    for text in df["text"]
    for token in tokenize(text)
})

itos = ["[PAD]", "[CLS]", "[UNK]"] + all_tokens
stoi = {token: idx for idx, token in enumerate(itos)}

MAX_LEN = 6

def encode(text):
    tokens = ["[CLS]"] + tokenize(text)
    ids = [stoi.get(token, stoi["[UNK]"]) for token in tokens][:MAX_LEN]
    mask = [1] * len(ids)

    ids += [stoi["[PAD]"]] * (MAX_LEN - len(ids))
    mask += [0] * (MAX_LEN - len(mask))

    return ids, mask, tokens[:MAX_LEN]

print("Tamaño del vocabulario:", len(itos))
display(pd.DataFrame({"id": range(len(itos)), "token": itos}))

In [ ]:
example_text = "vpn no funciona"
example_ids, example_mask, example_tokens = encode(example_text)

print("Texto:", example_text)
print("Tokens:", example_tokens)
print("IDs:", example_ids)
print("Mask:", example_mask)

display(pd.DataFrame({
    "position": range(MAX_LEN),
    "token": example_tokens + ["[PAD]"] * (MAX_LEN - len(example_tokens)),
    "id": example_ids,
    "mask": example_mask,
}))

La máscara vale `1` para tokens reales y `0` para padding. Más adelante impediremos que attention utilice los `[PAD]` como información útil.

## 4. Embeddings: IDs → vectores

Un ID como `16` no tiene significado geométrico por sí mismo.

Usamos una tabla de embeddings que transforma cada token en un vector de `d_model` números.

En este laboratorio:

```text
d_model = 16
```

El embedding es **entrenable**: sus valores también se modificarán durante el aprendizaje.

In [ ]:
D_MODEL = 16

torch.manual_seed(SEED)
demo_token_embedding = nn.Embedding(
    num_embeddings=len(itos),
    embedding_dim=D_MODEL,
    padding_idx=stoi["[PAD]"],
)

ids_t = torch.tensor([example_ids], dtype=torch.long)
token_vectors = demo_token_embedding(ids_t)

print("Shape:", tuple(token_vectors.shape))
print("Interpretación: batch=1, tokens=6, dimensiones=16")

rows = []
tokens_with_pad = example_tokens + ["[PAD]"] * (MAX_LEN - len(example_tokens))
for pos, token in enumerate(tokens_with_pad):
    rows.append({
        "position": pos,
        "token": token,
        "id": example_ids[pos],
        "embedding_first_4": np.round(
            token_vectors[0, pos, :4].detach().numpy(), 4
        ).tolist(),
    })

display(pd.DataFrame(rows))

El embedding de un token es inicialmente el mismo independientemente de la frase en la que aparezca. Todavía no es una representación contextual.

La contextualización aparecerá cuando los tokens puedan intercambiar información mediante attention.

## 5. La posición también importa

Self-attention, por sí sola, no conoce el orden.

Por eso añadimos un embedding de posición:

```text
representación inicial
=
token embedding
+
position embedding
```

Así el modelo puede distinguir, por ejemplo, el mismo token situado en posiciones distintas.

In [ ]:
torch.manual_seed(SEED)
demo_position_embedding = nn.Embedding(MAX_LEN, D_MODEL)

positions = torch.arange(MAX_LEN).unsqueeze(0)
position_vectors = demo_position_embedding(positions)

x = token_vectors + position_vectors

print("Token embeddings:", tuple(token_vectors.shape))
print("Position embeddings:", tuple(position_vectors.shape))
print("Entrada a attention:", tuple(x.shape))

display(pd.DataFrame({
    "token": tokens_with_pad,
    "pos": range(MAX_LEN),
    "token_dim0": token_vectors[0, :, 0].detach().numpy(),
    "position_dim0": position_vectors[0, :, 0].detach().numpy(),
    "sum_dim0": x[0, :, 0].detach().numpy(),
}).round(4))

## 6. Q, K y V

Cada representación de entrada se proyecta en tres vectores:

```text
Q = Query
K = Key
V = Value
```

Una intuición útil:

- **Query:** ¿qué estoy buscando?
- **Key:** ¿qué información represento?
- **Value:** ¿qué información aportaré si recibo atención?

En un Transformer real estas proyecciones se aprenden.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)
        self.d_model = d_model

    def forward(self, x, mask):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = Q @ K.transpose(-2, -1)
        scaled_scores = scores / math.sqrt(self.d_model)

        scaled_scores = scaled_scores.masked_fill(
            mask[:, None, :] == 0,
            -1e9,
        )

        attention = torch.softmax(scaled_scores, dim=-1)
        context = attention @ V
        output = self.out(context)

        return output, attention, Q, K, V, scores, scaled_scores, context

torch.manual_seed(SEED)
demo_attention = SelfAttention(D_MODEL)

mask_t = torch.tensor([example_mask], dtype=torch.float32)

with torch.no_grad():
    (
        demo_output,
        demo_weights,
        Q,
        K,
        V,
        raw_scores,
        scaled_scores,
        context,
    ) = demo_attention(x, mask_t)

print("Q:", tuple(Q.shape))
print("K:", tuple(K.shape))
print("V:", tuple(V.shape))
print("Scores QKᵀ:", tuple(raw_scores.shape))
print("Attention:", tuple(demo_weights.shape))
print("Context:", tuple(context.shape))

Cada uno de nuestros 6 tokens produce un Query, un Key y un Value de 16 dimensiones.

Al multiplicar:

```text
Q · Kᵀ
```

obtenemos una matriz `6 × 6`: cada token puede comparar su Query con el Key de todos los tokens.

## 7. Scores, escala y softmax

Los scores se calculan como:

```text
QKᵀ / √d_k
```

Dividir por `√d_k` evita que los productos escalares crezcan demasiado al aumentar la dimensión.

Después aplicamos `softmax` por filas.

El resultado son pesos positivos que suman aproximadamente `1` en cada fila.

In [ ]:
valid_len = len(example_tokens)
visible_tokens = tokens_with_pad[:valid_len]

score_df = pd.DataFrame(
    scaled_scores[0, :valid_len, :valid_len].detach().numpy(),
    index=visible_tokens,
    columns=visible_tokens,
)

weight_df = pd.DataFrame(
    demo_weights[0, :valid_len, :valid_len].detach().numpy(),
    index=visible_tokens,
    columns=visible_tokens,
)

print("Scores escalados:")
display(score_df.round(3))

print("Pesos después de softmax:")
display(weight_df.round(3))

print("Suma por fila:")
display(weight_df.sum(axis=1).round(6).to_frame("sum"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(weight_df.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(visible_tokens)), visible_tokens, rotation=45, ha="right")
ax.set_yticks(range(len(visible_tokens)), visible_tokens)
ax.set_xlabel("Key — token al que se atiende")
ax.set_ylabel("Query — token que busca información")
ax.set_title("Matriz de atención — pesos iniciales sin entrenar")

for i in range(len(visible_tokens)):
    for j in range(len(visible_tokens)):
        ax.text(j, i, f"{weight_df.iloc[i, j]:.2f}", ha="center", va="center")

fig.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

### Una advertencia esencial

Estos pesos todavía proceden de proyecciones **sin entrenar**.

La matriz sirve para entender la mecánica:

```text
Q/K
→ scores
→ softmax
→ pesos
```

pero **no debemos buscar todavía una interpretación semántica profunda**.

Un patrón de attention solo resulta útil después de que el modelo haya aprendido una tarea.

## 8. Context: mezclar información de otros tokens

Los pesos de attention se utilizan para combinar los Values:

```text
context = attention @ V
```

Cada token obtiene así una nueva representación construida a partir de otros tokens de la secuencia.

In [ ]:
context_df = pd.DataFrame({
    "token": visible_tokens,
    "V_dim0": V[0, :valid_len, 0].detach().numpy(),
    "context_dim0": context[0, :valid_len, 0].detach().numpy(),
})

display(context_df.round(4))

Hasta aquí hemos abierto la fórmula de self-attention.

Ahora entrenaremos una versión mínima de un bloque Transformer para que esas proyecciones Q/K/V aprendan algo útil.

## 9. Preparar datos para un Tiny Transformer

Nuestro pequeño problema tendrá 112 ejemplos:

```text
0 = healthy
1 = incident
```

Dividimos train y validation de forma estratificada.

In [ ]:
X_ids = []
X_masks = []
labels = []

for text, label in samples:
    ids, mask, _ = encode(text)
    X_ids.append(ids)
    X_masks.append(mask)
    labels.append(label)

X_ids = np.array(X_ids)
X_masks = np.array(X_masks)
labels = np.array(labels)

indices = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.25,
    stratify=labels,
    random_state=SEED,
)

X_train = torch.tensor(X_ids[train_idx], dtype=torch.long, device=DEVICE)
M_train = torch.tensor(X_masks[train_idx], dtype=torch.float32, device=DEVICE)
y_train = torch.tensor(labels[train_idx], dtype=torch.long, device=DEVICE)

X_val = torch.tensor(X_ids[val_idx], dtype=torch.long, device=DEVICE)
M_val = torch.tensor(X_masks[val_idx], dtype=torch.float32, device=DEVICE)
y_val = torch.tensor(labels[val_idx], dtype=torch.long, device=DEVICE)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Train clases:", torch.bincount(y_train))
print("Validation clases:", torch.bincount(y_val))

## 10. Nuestro Tiny Transformer

Construiremos un único bloque con:

```text
token embeddings
+
position embeddings
        ↓
self-attention
        ↓
residual + LayerNorm
        ↓
feed-forward
        ↓
residual + LayerNorm
        ↓
representación [CLS]
        ↓
clasificador
```

Es una versión mínima, de **una sola cabeza de atención** y una sola capa.

Los Transformers reales suelen usar múltiples heads y muchas capas repetidas.

In [ ]:
class TinyTransformerClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=16,
        ff_dim=32,
        max_len=MAX_LEN,
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=stoi["[PAD]"],
        )
        self.position_embedding = nn.Embedding(max_len, d_model)

        self.attention = SelfAttention(d_model)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, ids, mask, return_details=False):
        positions = torch.arange(
            ids.size(1),
            device=ids.device,
        ).unsqueeze(0).expand_as(ids)

        token_vectors = self.token_embedding(ids)
        x = token_vectors + self.position_embedding(positions)

        (
            attention_output,
            attention_weights,
            Q,
            K,
            V,
            raw_scores,
            scaled_scores,
            context,
        ) = self.attention(x, mask)

        x1 = self.norm1(x + attention_output)
        contextual = self.norm2(x1 + self.ff(x1))

        logits = self.classifier(contextual[:, 0, :])

        if return_details:
            return logits, {
                "token_embedding": token_vectors,
                "input_with_position": x,
                "Q": Q,
                "K": K,
                "V": V,
                "scaled_scores": scaled_scores,
                "attention": attention_weights,
                "context": context,
                "contextual": contextual,
            }

        return logits

torch.manual_seed(SEED)
model = TinyTransformerClassifier(len(itos)).to(DEVICE)

print(model)
print(
    "Parámetros entrenables:",
    sum(p.numel() for p in model.parameters()),
)

El modelo sigue siendo diminuto: solo unos pocos miles de parámetros.

El objetivo no es obtener un foundation model. Queremos observar cómo attention aprende una tarea sencilla.

## 11. Entrenar el Tiny Transformer

Usamos `CrossEntropyLoss` porque tenemos dos clases y el modelo devuelve dos logits.

Entrenaremos solo unas pocas decenas de épocas.

In [ ]:
torch.manual_seed(SEED)

model = TinyTransformerClassifier(len(itos)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

history = []

for epoch in range(41):
    model.train()

    optimizer.zero_grad()
    train_logits = model(X_train, M_train)
    train_loss = loss_fn(train_logits, y_train)
    train_loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        train_accuracy = (
            train_logits.argmax(dim=1) == y_train
        ).float().mean().item()

        val_logits = model(X_val, M_val)
        val_loss = loss_fn(val_logits, y_val)

        val_accuracy = (
            val_logits.argmax(dim=1) == y_val
        ).float().mean().item()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss.item(),
        "val_loss": val_loss.item(),
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
    })

history_df = pd.DataFrame(history)

display(
    history_df.loc[
        history_df["epoch"].isin([0, 1, 5, 10, 20, 40])
    ].round(4)
)

print(
    "Accuracy final validation:",
    f"{history_df.iloc[-1]['val_accuracy']:.1%}",
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], label="train")
ax.plot(history_df["epoch"], history_df["val_loss"], label="validation")
ax.set_title("Tiny Transformer — loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_accuracy"], label="train")
ax.plot(history_df["epoch"], history_df["val_accuracy"], label="validation")
ax.set_title("Tiny Transformer — accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 12. Inspeccionar attention después del entrenamiento

Ahora sí tiene sentido mirar los pesos.

Compararemos:

```text
vpn no funciona
```

con:

```text
vpn sí funciona
```

Las dos frases contienen `vpn` y `funciona`, pero una representa un incidente y la otra un servicio saludable.

In [ ]:
def inspect_sentence(text):
    ids, mask, tokens = encode(text)

    ids_t = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    mask_t = torch.tensor([mask], dtype=torch.float32, device=DEVICE)

    model.eval()
    with torch.no_grad():
        logits, details = model(
            ids_t,
            mask_t,
            return_details=True,
        )
        probabilities = torch.softmax(logits, dim=1)[0]

    full_tokens = tokens + ["[PAD]"] * (MAX_LEN - len(tokens))

    return {
        "text": text,
        "tokens": full_tokens,
        "ids": ids_t,
        "mask": mask_t,
        "probabilities": probabilities,
        "prediction": logits.argmax(dim=1).item(),
        "details": details,
        "valid_len": len(tokens),
    }

negative = inspect_sentence("vpn no funciona")
positive = inspect_sentence("vpn sí funciona")

comparison = pd.DataFrame([
    {
        "text": negative["text"],
        "P(healthy)": negative["probabilities"][0].item(),
        "P(incident)": negative["probabilities"][1].item(),
        "prediction": "incident" if negative["prediction"] == 1 else "healthy",
    },
    {
        "text": positive["text"],
        "P(healthy)": positive["probabilities"][0].item(),
        "P(incident)": positive["probabilities"][1].item(),
        "prediction": "incident" if positive["prediction"] == 1 else "healthy",
    },
])

display(comparison.style.format({
    "P(healthy)": "{:.1%}",
    "P(incident)": "{:.1%}",
}))

In [ ]:
def attention_dataframe(result):
    n = result["valid_len"]
    tokens = result["tokens"][:n]
    weights = (
        result["details"]["attention"][0, :n, :n]
        .detach()
        .cpu()
        .numpy()
    )

    return pd.DataFrame(
        weights,
        index=tokens,
        columns=tokens,
    )

neg_attn = attention_dataframe(negative)
pos_attn = attention_dataframe(positive)

print("vpn no funciona")
display(neg_attn.round(3))

print("vpn sí funciona")
display(pos_attn.round(3))

In [ ]:
for title, attn_df in [
    ("vpn no funciona", neg_attn),
    ("vpn sí funciona", pos_attn),
]:
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(attn_df.values, vmin=0, vmax=1, cmap="viridis")

    tokens = list(attn_df.columns)
    ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    ax.set_title(f"Attention aprendida — {title}")

    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(
                j,
                i,
                f"{attn_df.iloc[i, j]:.2f}",
                ha="center",
                va="center",
            )

    fig.colorbar(im, ax=ax, label="attention weight")
    plt.tight_layout()
    plt.show()

### Cómo leer la matriz

Cada fila representa un **Query**.

Cada columna representa un **Key**.

Por ejemplo, la fila `[CLS]` nos permite observar qué tokens utiliza la representación destinada a clasificar la frase.

Importante:

> Un peso alto de attention no es automáticamente una explicación causal completa de la predicción.

Es una señal observable del mecanismo, no una prueba definitiva de “por qué pensó eso”.

## 13. El mismo token puede acabar con otra representación

El token:

```text
funciona
```

aparece en ambas frases y en la misma posición:

```text
vpn no funciona
vpn sí funciona
```

Antes de attention, su **token embedding** es exactamente el mismo.

Después del bloque Transformer, su representación contextual puede ser muy diferente.

In [ ]:
funciona_position = 3

raw_negative = negative["details"]["token_embedding"][0, funciona_position]
raw_positive = positive["details"]["token_embedding"][0, funciona_position]

ctx_negative = negative["details"]["contextual"][0, funciona_position]
ctx_positive = positive["details"]["contextual"][0, funciona_position]

raw_cosine = F.cosine_similarity(
    raw_negative,
    raw_positive,
    dim=0,
).item()

contextual_cosine = F.cosine_similarity(
    ctx_negative,
    ctx_positive,
    dim=0,
).item()

raw_distance = torch.norm(raw_negative - raw_positive).item()
contextual_distance = torch.norm(ctx_negative - ctx_positive).item()

display(pd.DataFrame([
    {
        "representation": "Token embedding antes de attention",
        "cosine_similarity": raw_cosine,
        "euclidean_distance": raw_distance,
    },
    {
        "representation": "Representación contextual después del bloque",
        "cosine_similarity": contextual_cosine,
        "euclidean_distance": contextual_distance,
    },
]).round(4))

Este es uno de los conceptos esenciales de los Transformers:

```text
embedding de token
→ identidad léxica inicial

representación contextual
→ token + posición + información de otros tokens
```

Por eso una misma palabra puede acabar representada de forma distinta según el contexto.

## 14. Q/K/V del modelo entrenado

Podemos abrir también las proyecciones aprendidas para una frase.

No intentaremos interpretar cada dimensión individualmente.

Lo importante es comprobar el flujo y las shapes.

In [ ]:
details = negative["details"]
n = negative["valid_len"]

summary = pd.DataFrame([
    {"tensor": "input_with_position", "shape": tuple(details["input_with_position"][:, :n].shape)},
    {"tensor": "Q", "shape": tuple(details["Q"][:, :n].shape)},
    {"tensor": "K", "shape": tuple(details["K"][:, :n].shape)},
    {"tensor": "V", "shape": tuple(details["V"][:, :n].shape)},
    {"tensor": "scaled_scores", "shape": tuple(details["scaled_scores"][:, :n, :n].shape)},
    {"tensor": "attention", "shape": tuple(details["attention"][:, :n, :n].shape)},
    {"tensor": "context", "shape": tuple(details["context"][:, :n].shape)},
    {"tensor": "contextual", "shape": tuple(details["contextual"][:, :n].shape)},
])

display(summary)

## 15. ¿Qué tiene de Transformer nuestro modelo?

Nuestro Tiny Transformer ya contiene varias piezas fundamentales:

```text
token embeddings
position embeddings
self-attention
Q / K / V
scaled dot-product
softmax
residual connection
LayerNorm
feed-forward network
```

Pero sigue siendo muchísimo más simple que un LLM.

## 16. Qué falta para parecerse a un LLM moderno

Entre otras cosas:

- muchas capas apiladas;
- múltiples attention heads;
- un vocabulario/tokenizer mucho mayor;
- cientos de millones o miles de millones de parámetros;
- entrenamiento con enormes corpus;
- en modelos generativos, objetivo de predicción del siguiente token;
- causal mask para impedir mirar tokens futuros durante generación;
- técnicas avanzadas de optimización y distribución.

No necesitamos implementar todo eso para comprender el mecanismo base.

## 17. Qué puedes modificar

Prueba cambios controlados:

- cambia `D_MODEL`;
- cambia `ff_dim`;
- reduce o aumenta las épocas;
- modifica una frase manteniendo el vocabulario;
- compara `vpn no funciona` con `crm no funciona`;
- compara `vpn sí funciona` con `vpn funciona`;
- observa la fila `[CLS]` de attention;
- observa cómo cambia la representación contextual.

Pregunta siempre:

```text
¿cambia la predicción?
¿cambia attention?
¿cambia la representación contextual?
¿qué parte es mecánica y qué parte fue aprendida?
```

## 18. Qué debes llevarte

Self-attention puede resumirse como:

```text
representaciones
   ↓
Q, K, V
   ↓
QKᵀ / √d_k
   ↓
softmax
   ↓
pesos de attention
   ↓
combinación de V
   ↓
representaciones contextualizadas
```

La idea clave no es memorizar las letras.

Es entender que cada token puede construir una nueva representación utilizando información de otros tokens de la secuencia.

## 19. Preguntas de reflexión

1. ¿Por qué un token ID no debe interpretarse como una magnitud?
2. ¿Qué hace un embedding?
3. ¿Por qué añadimos información posicional?
4. ¿Qué significan intuitivamente Q, K y V?
5. ¿Por qué `QKᵀ` produce una matriz token × token?
6. ¿Por qué dividimos por `√d_k`?
7. ¿Qué aporta softmax?
8. ¿Por qué cada fila de attention suma aproximadamente 1?
9. ¿Qué función cumple la máscara de padding?
10. ¿Por qué no debemos interpretar semánticamente attention sin entrenar?
11. ¿Qué significa que `[CLS]` atienda a ciertos tokens?
12. ¿Por qué attention no es automáticamente una explicación causal?
13. ¿Qué diferencia hay entre token embedding y representación contextual?
14. ¿Cómo puede `funciona` tener el mismo embedding inicial pero una representación final distinta?
15. ¿Qué aportan las conexiones residuales?
16. ¿Qué aporta la feed-forward network?
17. ¿Por qué nuestro Tiny Transformer no es un LLM?
18. ¿Qué es multi-head attention?
19. ¿Qué sería una causal mask?
20. ¿Qué piezas de este laboratorio siguen existiendo, a mucha mayor escala, en un LLM moderno?

## 20. Idea final

En M03-P01 vimos que una red aprende modificando parámetros mediante gradientes.

En M03-P02 hemos visto qué puede aprender uno de los mecanismos centrales de los Transformers:

```text
qué información combinar
+
cuánto combinarla
+
cómo convertirla en una representación contextual
```

Una palabra ya no tiene por qué estar representada únicamente por “qué palabra es”.

Puede estar representada también por:

```text
qué palabra es
+
dónde aparece
+
qué otros tokens la rodean
+
qué relaciones ha aprendido el modelo
```

Ese salto de **embedding estático** a **representación contextual** es una de las ideas fundamentales que necesitaremos para comprender los modelos generativos modernos.